In [ ]:
import os
import glob
import math
import random
import shutil
import subprocess
import base64
import zlib
import re
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MODEL_VARIANT = "hat_s"
SCALE = 4

TOTAL_TARGET_ITER = 50000
ITERS_PER_SESSION = 30000
AUTO_RESUME_FROM_INPUT = False

BATCH_SIZE = 4
GT_SIZE = 256
LEARNING_RATE = 1e-4

ADAM_BETAS = [0.9, 0.99]
WEIGHT_DECAY = 0.0
EMA_DECAY = 0.999

USE_CONSTANT_LR = False
WARMUP_ITER = -1

USE_CLEAN6_IF_AVAILABLE = True

OFFICIAL_HAT_S_PATH = (
    "/kaggle/input/datasets/djokester/hat-pre-trained-weights-for-super-resolution/"
    "HAT_Pretrained_Models/HAT-S_SRx4.pth"
)

PRETRAIN_PATH = OFFICIAL_HAT_S_PATH
PARAM_KEY_G = "params_ema"
STRICT_LOAD_G = True

CLEAN_PSNR_THRESHOLD = 18.0
USE_CLEAN_SYMLINK_DATA = True

USE_PREVIOUS_CLEAN_REPORT = True
PREVIOUS_CLEAN_REPORT = "/kaggle/input/datasets/username/clean_pair_report.csv"

FORCE_REBUILD_CLEAN_DATA = False

USE_PUBLIC_SUSPECT_TXT = True
PUBLIC_SUSPECT_PATTERNS = [
    "/kaggle/input/**/suspects*.txt",
    "/kaggle/input/**/suspects_to_choose_threshold.txt",
]

ENABLE_HOLDOUT_VAL = False

SAVE_CHECKPOINT_FREQ = 1000
EXPORT_RESUME_PACKAGE = True

FORCE_DELETE_EXISTING_EXPERIMENT = True

RUN_INFERENCE_EACH_SESSION = True
FORCE_INFERENCE_NOW = True

RUN_TTA_SUBMISSION = True
USE_TTA = True
TTA_MODES = list(range(8))


USE_HALF = False
ENCODE_BGR = True
WINDOW_SIZE = 16

WORK_DIR = "/kaggle/working"
HAT_DIR = f"{WORK_DIR}/HAT"

BASE_EXP_NAME = "vg_reference_33p5_hat_s_60k"
EXP_NAME = f"{BASE_EXP_NAME}_{MODEL_VARIANT}"

CLEAN_HR = f"{WORK_DIR}/clean_train/hr"
CLEAN_LR = f"{WORK_DIR}/clean_train/lr"
REPORT_CSV = f"{WORK_DIR}/clean_pair_report.csv"
CLEAN_CONFIG_JSON = f"{WORK_DIR}/clean_config.json"

os.makedirs(WORK_DIR, exist_ok=True)

COMP_CANDIDATES = [
    "/kaggle/input/competitions/super-resolution-in-video-games",
    "/kaggle/input/super-resolution-in-video-games",
]

COMP_DIR = None

for p in COMP_CANDIDATES:
    if (
        os.path.isdir(f"{p}/train/hr") and
        os.path.isdir(f"{p}/train/lr") and
        os.path.isdir(f"{p}/test/lr") and
        os.path.isfile(f"{p}/sample_submission.csv")
    ):
        COMP_DIR = p
        break

if COMP_DIR is None:
    for sub_path in glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True):
        root = os.path.dirname(sub_path)
        if (
            os.path.isdir(f"{root}/train/hr") and
            os.path.isdir(f"{root}/train/lr") and
            os.path.isdir(f"{root}/test/lr")
        ):
            COMP_DIR = root
            break

if COMP_DIR is None:
    raise FileNotFoundError(
        "Cannot find competition data. Add the competition dataset to Kaggle input."
    )

TRAIN_HR = f"{COMP_DIR}/train/hr"
TRAIN_LR = f"{COMP_DIR}/train/lr"
TEST_LR = f"{COMP_DIR}/test/lr"
SAMPLE_SUB = f"{COMP_DIR}/sample_submission.csv"

print("Configuration loaded.")
print("COMP_DIR                    =", COMP_DIR)
print("EXP_NAME                    =", EXP_NAME)
print("MODEL_VARIANT               =", MODEL_VARIANT)
print("TOTAL_TARGET_ITER           =", TOTAL_TARGET_ITER)
print("ITERS_PER_SESSION           =", ITERS_PER_SESSION)
print("AUTO_RESUME_FROM_INPUT      =", AUTO_RESUME_FROM_INPUT)
print("FORCE_DELETE_EXISTING_EXPERIMENT =", FORCE_DELETE_EXISTING_EXPERIMENT)
print("BATCH_SIZE                  =", BATCH_SIZE)
print("GT_SIZE                     =", GT_SIZE)
print("LEARNING_RATE               =", LEARNING_RATE)
print("PRETRAIN_PATH               =", PRETRAIN_PATH)
print("USE_PREVIOUS_CLEAN_REPORT   =", USE_PREVIOUS_CLEAN_REPORT)
print("PREVIOUS_CLEAN_REPORT       =", PREVIOUS_CLEAN_REPORT)
print("RUN_INFERENCE_EACH_SESSION  =", RUN_INFERENCE_EACH_SESSION)
print("RUN_TTA_SUBMISSION          =", RUN_TTA_SUBMISSION)
print("USE_HALF                    =", USE_HALF)

In [ ]:
import os
import sys
import glob
import shutil
import subprocess
from pathlib import Path

def run(cmd, cwd=None, check=True):
    print("\n$ " + cmd)
    return subprocess.run(cmd, shell=True, cwd=cwd, check=check)

run(
    "pip install -q pyyaml opencv-python pillow pandas tqdm numpy scipy scikit-image einops timm requests",
    check=False
)


if os.path.exists(HAT_DIR):
    print("HAT repo already exists:", HAT_DIR)
else:
    run(f"git clone --depth 1 https://github.com/XPixelGroup/HAT.git {HAT_DIR}")

run("pip install -q -r requirements.txt", cwd=HAT_DIR, check=False)
run("python setup.py develop", cwd=HAT_DIR, check=False)

if HAT_DIR not in sys.path:
    sys.path.insert(0, HAT_DIR)

candidate_paths = glob.glob("/usr/local/lib/python*/dist-packages/basicsr/data/degradations.py")
candidate_paths += glob.glob("/opt/conda/lib/python*/site-packages/basicsr/data/degradations.py")

for p in candidate_paths:
    if os.path.exists(p):
        txt = Path(p).read_text(encoding="utf-8")
        old = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
        new = "from torchvision.transforms.functional import rgb_to_grayscale"

        if old in txt:
            Path(p).write_text(txt.replace(old, new), encoding="utf-8")
            print("Patched BasicSR:", p)

try:
    import torchvision.transforms

    transforms_dir = os.path.dirname(torchvision.transforms.__file__)
    shim_path = os.path.join(transforms_dir, "functional_tensor.py")

    if not os.path.exists(shim_path):
        Path(shim_path).write_text(
            "from .functional import rgb_to_grayscale\n",
            encoding="utf-8"
        )
        print("Created shim:", shim_path)
    else:
        print("Torchvision shim already exists:", shim_path)

except Exception as e:
    print("Torchvision shim warning:", repr(e))


MODEL_DIR = "/kaggle/working/downloaded_models"
os.makedirs(MODEL_DIR, exist_ok=True)

def find_file(filename, roots=("/kaggle/input", "/kaggle/working")):
    matches = []
    for root in roots:
        if os.path.exists(root):
            matches += glob.glob(os.path.join(root, "**", filename), recursive=True)
    return sorted(set(matches))

def download_file(urls, dst_path, min_size_mb=10):
    dst_path = Path(dst_path)
    dst_path.parent.mkdir(parents=True, exist_ok=True)

    if dst_path.exists() and dst_path.stat().st_size > min_size_mb * 1024 * 1024:
        print("Already downloaded:", dst_path)
        print("Size MB:", round(dst_path.stat().st_size / 1024 / 1024, 2))
        return str(dst_path)

    for url in urls:
        print("\nTrying download:")
        print(url)

        if dst_path.exists():
            dst_path.unlink()

        # wget is usually available on Kaggle.
        ret = subprocess.run(
            f'wget -q --show-progress -O "{dst_path}" "{url}"',
            shell=True
        )

        if dst_path.exists() and dst_path.stat().st_size > min_size_mb * 1024 * 1024:
            print("Downloaded successfully:", dst_path)
            print("Size MB:", round(dst_path.stat().st_size / 1024 / 1024, 2))
            return str(dst_path)

        print("Download failed or file too small.")

    raise FileNotFoundError(
        f"Could not download checkpoint to {dst_path}. "
        "If Kaggle Internet is disabled, enable Internet or upload the .pth as a Kaggle Dataset."
    )
if MODEL_VARIANT == "hat_s":
    official_name = "HAT-S_SRx4.pth"
    official_dst = f"{MODEL_DIR}/HAT-S_SRx4.pth"
    official_urls = [
        "https://huggingface.co/jaideepsingh/upscale_models/resolve/main/HAT/HAT-S_SRx4.pth?download=true",
        "https://huggingface.co/Acly/hat/resolve/main/HAT-S_SRx4.pth?download=true",
    ]

elif MODEL_VARIANT == "full_hat":
    official_name = "HAT_SRx4.pth"
    official_dst = f"{MODEL_DIR}/HAT_SRx4.pth"
    official_urls = [
        "https://huggingface.co/jaideepsingh/upscale_models/resolve/main/HAT/HAT_SRx4.pth?download=true",
    ]

else:
    raise ValueError("MODEL_VARIANT must be 'hat_s' or 'full_hat'.")

found_ckpt = None

if MODEL_VARIANT == "hat_s" and USE_CLEAN6_IF_AVAILABLE:
    clean6_matches = find_file("HAT-S_SRx4-clean-6.pth")

    if clean6_matches:
        found_ckpt = clean6_matches[0]
        print("Using accessible clean-6 checkpoint:")
        print(found_ckpt)
    else:
        print("clean-6 checkpoint not found.")

if found_ckpt is None:
    official_matches = find_file(official_name)

    if official_matches:
        found_ckpt = official_matches[0]
        print("Using existing official checkpoint:")
        print(found_ckpt)
    else:
        print(f"{official_name} not found in /kaggle/input or /kaggle/working.")
        print("Trying direct download...")

        found_ckpt = download_file(
            official_urls,
            official_dst,
            min_size_mb=50
        )

PRETRAIN_PATH = found_ckpt
MODEL_PATH = PRETRAIN_PATH

PARAM_KEY_G = "params_ema"
STRICT_LOAD_G = True

if not os.path.exists(PRETRAIN_PATH):
    print("\nAll visible .pth files under /kaggle/input:")
    for p in sorted(glob.glob("/kaggle/input/**/*.pth", recursive=True)):
        print(" -", p)

    raise FileNotFoundError(f"PRETRAIN_PATH not found: {PRETRAIN_PATH}")

print("\nCheckpoint ready.")
print("MODEL_VARIANT =", MODEL_VARIANT)
print("PRETRAIN_PATH =", PRETRAIN_PATH)
print("MODEL_PATH    =", MODEL_PATH)
print("PARAM_KEY_G   =", PARAM_KEY_G)
print("Exists        =", os.path.exists(PRETRAIN_PATH))
print("Size MB       =", round(os.path.getsize(PRETRAIN_PATH) / 1024 / 1024, 2))

In [ ]:
import cv2
import json

def safe_symlink(src, dst):
    if os.path.exists(dst):
        return
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)

def calc_psnr_bgr(a, b):
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    mse = np.mean((a - b) ** 2)
    if mse <= 1e-12:
        return 99.0
    return 20.0 * math.log10(255.0 / math.sqrt(float(mse)))

def load_public_suspects():
    suspects = set()
    if not USE_PUBLIC_SUSPECT_TXT:
        return suspects
    for pat in PUBLIC_SUSPECT_PATTERNS:
        for fp in glob.glob(pat, recursive=True):
            try:
                for line in Path(fp).read_text(encoding="utf-8").splitlines():
                    line = line.strip()
                    if line:
                        suspects.add(os.path.basename(line))
                print("Loaded suspects:", fp, len(suspects))
            except Exception as e:
                print("Could not read suspect file:", fp, repr(e))
    return suspects

hr_files = {os.path.basename(p): p for p in glob.glob(os.path.join(TRAIN_HR, "*"))}
lr_files = {os.path.basename(p): p for p in glob.glob(os.path.join(TRAIN_LR, "*"))}
common = sorted(set(hr_files) & set(lr_files))
print("Common train pairs:", len(common))

if os.path.exists(PREVIOUS_CLEAN_REPORT):
    print("Using previous clean report:", PREVIOUS_CLEAN_REPORT)
    report = pd.read_csv(PREVIOUS_CLEAN_REPORT)
else:
    print("Previous report not found. Computing clean report now.")
    suspects = load_public_suspects()
    rows = []
    for name in tqdm(common, desc="Checking PSNR pairs"):
        hp, lp = hr_files[name], lr_files[name]
        keep, reason, psnr = True, "keep", None
        if name in suspects:
            keep, reason = False, "public_suspect_list"
        else:
            hr = cv2.imread(hp, cv2.IMREAD_COLOR)
            lr = cv2.imread(lp, cv2.IMREAD_COLOR)
            if hr is None or lr is None:
                keep, reason = False, "read_error"
            else:
                lr_up = cv2.resize(lr, (hr.shape[1], hr.shape[0]), interpolation=cv2.INTER_CUBIC)
                psnr = calc_psnr_bgr(hr, lr_up)
                if psnr < CLEAN_PSNR_THRESHOLD:
                    keep, reason = False, f"low_psnr<{CLEAN_PSNR_THRESHOLD}"
        rows.append({"filename": name, "keep": keep, "reason": reason, "bicubic_psnr": psnr})
    report = pd.DataFrame(rows)
    report.to_csv(REPORT_CSV, index=False)

for d in [CLEAN_HR, CLEAN_LR]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

keep_col = report["keep"]
if keep_col.dtype != bool:
    keep_mask = keep_col.astype(str).str.lower().isin(["true", "1", "yes"])
else:
    keep_mask = keep_col

keep_names = report.loc[keep_mask, "filename"].astype(str).map(os.path.basename).tolist()
kept = 0
missing = 0
for name in tqdm(keep_names, desc="Building clean symlinks"):
    if name in hr_files and name in lr_files:
        safe_symlink(hr_files[name], os.path.join(CLEAN_HR, name))
        safe_symlink(lr_files[name], os.path.join(CLEAN_LR, name))
        kept += 1
    else:
        missing += 1

report.to_csv(REPORT_CSV, index=False)
with open(CLEAN_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump({"threshold": CLEAN_PSNR_THRESHOLD, "source": PREVIOUS_CLEAN_REPORT}, f, indent=2)

TRAIN_HR_USED = CLEAN_HR
TRAIN_LR_USED = CLEAN_LR

print("Clean images:", kept, "missing:", missing)
print("Training HR:", TRAIN_HR_USED)
print("Training LR:", TRAIN_LR_USED)

In [ ]:
def iter_from_path(p):
    nums = re.findall(r"\d+", Path(p).stem)
    return int(nums[-1]) if nums else -1

def find_latest_resume_package(exp_name):
    state_patterns = [
        f"/kaggle/input/**/resume_package/{exp_name}/training_states/*.state",
        f"/kaggle/input/**/{exp_name}/training_states/*.state",
    ]
    states = []
    for pat in state_patterns:
        states += glob.glob(pat, recursive=True)
    states = sorted(set(states), key=iter_from_path)
    if not states:
        return None, None
    state_path = states[-1]
    it = iter_from_path(state_path)
    root = Path(state_path).parents[1]
    model_candidates = [
        str(root / "models" / f"net_g_{it}.pth"),
        str(root / "models" / "net_g_latest.pth"),
    ]
    model_path = next((p for p in model_candidates if os.path.exists(p)), None)
    return state_path, model_path

RESUME_STATE_PATH = None
RESUME_MODEL_PATH = None
RESUME_ITER = 0
AUTO_RESUME = False

if AUTO_RESUME_FROM_INPUT:
    st, mp = find_latest_resume_package(EXP_NAME)
    if st and mp:
        RESUME_STATE_PATH = st
        RESUME_MODEL_PATH = mp
        RESUME_ITER = iter_from_path(st)
        AUTO_RESUME = True
        print("Found resume state:", RESUME_STATE_PATH)
        print("Found resume model:", RESUME_MODEL_PATH)
        print("RESUME_ITER:", RESUME_ITER)
    else:
        print("No previous resume package found. Starting from pretrained checkpoint.")

SESSION_END_ITER = min(TOTAL_TARGET_ITER, RESUME_ITER + ITERS_PER_SESSION)
TOTAL_ITER = SESSION_END_ITER

if RESUME_ITER >= TOTAL_TARGET_ITER:
    RUN_TRAINING = False
    print("Already reached target iter. Training will be skipped.")
else:
    RUN_TRAINING = True

num_train_images = len(glob.glob(os.path.join(TRAIN_HR_USED, "*")))
steps_per_epoch = math.ceil(num_train_images / BATCH_SIZE)
print("num_train_images:", num_train_images)
print("steps_per_epoch:", steps_per_epoch)
print("RESUME_ITER:", RESUME_ITER)
print("This session trains to TOTAL_ITER:", TOTAL_ITER)
print("RUN_TRAINING:", RUN_TRAINING)

In [ ]:
exp_dir = Path(HAT_DIR) / "experiments" / EXP_NAME
model_dir = exp_dir / "models"
state_dir = exp_dir / "training_states"
vis_dir = exp_dir / "visualization"

for d in [exp_dir, model_dir, state_dir, vis_dir]:
    d.mkdir(parents=True, exist_ok=True)

if AUTO_RESUME:
    work_model = model_dir / f"net_g_{RESUME_ITER}.pth"
    work_latest = model_dir / "net_g_latest.pth"
    work_state = state_dir / f"{RESUME_ITER}.state"
    shutil.copy2(RESUME_MODEL_PATH, work_model)
    shutil.copy2(RESUME_MODEL_PATH, work_latest)

    import torch
    state = torch.load(RESUME_STATE_PATH, map_location="cpu")
    old_epoch = int(state.get("epoch", 0))
    state["iter"] = RESUME_ITER
    total_epochs = math.ceil(TOTAL_ITER / steps_per_epoch)
    new_epoch = max(0, min(int(RESUME_ITER // steps_per_epoch), max(0, total_epochs - 1)))
    state["epoch"] = new_epoch
    torch.save(state, work_state)

    RESUME_STATE_PATH = str(work_state)
    PRETRAIN_NETWORK_G = str(work_model)
    print("Restored resume files.")
    print("old_epoch:", old_epoch, "new_epoch:", new_epoch)
    print("working model:", work_model)
    print("working state:", RESUME_STATE_PATH)
else:
    PRETRAIN_NETWORK_G = PRETRAIN_PATH
    print("Start from pretrained:", PRETRAIN_NETWORK_G)

In [ ]:
import yaml

base_yml = os.path.join(HAT_DIR, "options/train/train_HAT-S_SRx4_finetune_from_SRx2.yml")
if not os.path.exists(base_yml):
    raise FileNotFoundError(base_yml)

with open(base_yml, "r", encoding="utf-8") as f:
    opt = yaml.safe_load(f)

opt["name"] = EXP_NAME
opt["model_type"] = "HATModel"
opt["scale"] = 4
opt["num_gpu"] = 1
opt["manual_seed"] = SEED
opt["auto_resume"] = False
opt["pbar"] = True

opt["datasets"] = {}
opt["datasets"]["train"] = {
    "name": "SR in Video Games clean train",
    "type": "PairedImageDataset",
    "dataroot_gt": TRAIN_HR_USED,
    "dataroot_lq": TRAIN_LR_USED,
    "io_backend": {"type": "disk"},
    "gt_size": int(GT_SIZE),
    "use_hflip": True,
    "use_rot": True,
    "use_shuffle": True,
    "num_worker_per_gpu": 2,
    "batch_size_per_gpu": int(BATCH_SIZE),
    "dataset_enlarge_ratio": 1,
    "prefetch_mode": "cuda",
    "pin_memory": True,
    "phase": "train",
    "scale": 4,
}

opt.setdefault("path", {})
opt["path"]["pretrain_network_g"] = PRETRAIN_NETWORK_G
opt["path"]["param_key_g"] = "params" if AUTO_RESUME else PARAM_KEY_G
opt["path"]["strict_load_g"] = bool(STRICT_LOAD_G)
opt["path"]["resume_state"] = RESUME_STATE_PATH if AUTO_RESUME else None

opt.setdefault("train", {})
opt["train"]["total_iter"] = int(TOTAL_ITER)
opt["train"]["warmup_iter"] = int(WARMUP_ITER)
opt["train"]["ema_decay"] = float(EMA_DECAY)
opt["train"]["optim_g"] = {
    "type": "Adam",
    "lr": float(LEARNING_RATE),
    "weight_decay": float(WEIGHT_DECAY),
    "betas": [float(ADAM_BETAS[0]), float(ADAM_BETAS[1])],
}
opt["train"]["pixel_opt"] = {"type": "L1Loss", "loss_weight": 1.0, "reduction": "mean"}
opt["train"]["scheduler"] = {"type": "MultiStepRestartLR", "milestones": [10**12], "gamma": 1.0}

opt.setdefault("logger", {})
opt["logger"]["print_freq"] = 100
opt["logger"]["save_checkpoint_freq"] = int(SAVE_CHECKPOINT_FREQ)
opt["logger"]["use_tb_logger"] = False

out_yml = os.path.join(HAT_DIR, f"options/train/{EXP_NAME}.yml")
with open(out_yml, "w", encoding="utf-8") as f:
    yaml.safe_dump(opt, f, sort_keys=False)

print("Wrote:", out_yml)
print(yaml.safe_dump({
    "name": EXP_NAME,
    "pretrain_network_g": opt["path"]["pretrain_network_g"],
    "resume_state": opt["path"]["resume_state"],
    "total_iter": opt["train"]["total_iter"],
    "batch_size": opt["datasets"]["train"]["batch_size_per_gpu"],
    "lr": opt["train"]["optim_g"]["lr"],
    "scheduler": opt["train"]["scheduler"],
    "train_gt": opt["datasets"]["train"]["dataroot_gt"],
    "train_lq": opt["datasets"]["train"]["dataroot_lq"],
}, sort_keys=False))

In [ ]:
if RUN_TRAINING and TOTAL_ITER > 0:
    run(f"python hat/train.py -opt {out_yml}", cwd=HAT_DIR)
else:
    print("Training skipped.")

# Export resume package
if EXPORT_RESUME_PACKAGE:
    resume_dir = Path(WORK_DIR) / "resume_package" / EXP_NAME
    resume_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["models", "training_states"]:
        src = exp_dir / sub
        dst = resume_dir / sub
        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)
    if os.path.exists(out_yml):
        shutil.copy2(out_yml, resume_dir / "train_config.yml")
    shutil.copy2(REPORT_CSV, Path(WORK_DIR) / "clean_pair_report.csv")
    print("Exported resume package:", resume_dir)
    print("Current model files:")
    for p in sorted(glob.glob(str(resume_dir / "models" / "*.pth"))):
        print(" -", p)
    print("Current state files:")
    for p in sorted(glob.glob(str(resume_dir / "training_states" / "*.state"))):
        print(" -", p)

In [ ]:
if RUN_INFERENCE_EACH_SESSION or (TOTAL_ITER >= TOTAL_TARGET_ITER) or FORCE_INFERENCE_NOW:
    import torch
    import torch.nn.functional as F
    from PIL import Image
    import sys
    sys.path.insert(0, HAT_DIR)
    os.chdir(HAT_DIR)
    import hat.archs.hat_arch  # noqa
    from basicsr.archs import build_network

    with open(out_yml, "r", encoding="utf-8") as f:
        infer_opt = yaml.safe_load(f)
    model = build_network(infer_opt["network_g"]).float().eval().cuda()

    def latest_pth():
        pths = glob.glob(str(model_dir / "net_g_*.pth"))
        def key(p):
            name = Path(p).stem
            if "latest" in name:
                return 10**12
            nums = re.findall(r"\d+", name)
            return int(nums[-1]) if nums else -1
        if pths:
            return sorted(pths, key=key)[-1]
        return PRETRAIN_NETWORK_G

    ckpt_path = latest_pth()
    ckpt = torch.load(ckpt_path, map_location="cpu")
    state = ckpt.get("params_ema", ckpt.get("params", ckpt))
    model.load_state_dict(state, strict=True)
    model.float().eval().cuda()
    print("Inference checkpoint:", ckpt_path)

    def encode(img: np.ndarray) -> str:
        img_to_encode = img.astype(np.uint8).flatten()
        img_to_encode = np.append(img_to_encode, -1)
        cnt, rle = 1, []
        for i in range(1, img_to_encode.shape[0]):
            if img_to_encode[i] == img_to_encode[i - 1]:
                cnt += 1
                if cnt > 255:
                    rle += [int(img_to_encode[i - 1]), 255]
                    cnt = 1
            else:
                rle += [int(img_to_encode[i - 1]), cnt]
                cnt = 1
        compressed = zlib.compress(bytes(rle), zlib.Z_BEST_COMPRESSION)
        return str(base64.b64encode(compressed))

    ops_for_mode = {
        0: [],
        1: ["hflip"],
        2: ["vflip"],
        3: ["hflip", "vflip"],
        4: ["transpose"],
        5: ["transpose", "hflip"],
        6: ["transpose", "vflip"],
        7: ["transpose", "hflip", "vflip"],
    }
    def apply_ops(x, ops):
        for op in ops:
            if op == "hflip":
                x = torch.flip(x, dims=[-1])
            elif op == "vflip":
                x = torch.flip(x, dims=[-2])
            elif op == "transpose":
                x = x.transpose(-1, -2)
        return x
    def invert_ops(x, ops):
        for op in reversed(ops):
            if op == "hflip":
                x = torch.flip(x, dims=[-1])
            elif op == "vflip":
                x = torch.flip(x, dims=[-2])
            elif op == "transpose":
                x = x.transpose(-1, -2)
        return x

    @torch.no_grad()
    def forward_once(x):
        x = x.float()
        _, _, h, w = x.shape
        pad_h = (WINDOW_SIZE - h % WINDOW_SIZE) % WINDOW_SIZE
        pad_w = (WINDOW_SIZE - w % WINDOW_SIZE) % WINDOW_SIZE
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")
        y = model(x.float())
        return y[:, :, :h * 4, :w * 4].float()

    @torch.no_grad()
    def predict_tensor(x):
        if not RUN_TTA_SUBMISSION:
            return forward_once(x)
        outs = []
        for m in TTA_MODES:
            ops = ops_for_mode[m]
            ya = forward_once(apply_ops(x, ops))
            outs.append(invert_ops(ya, ops))
        return torch.stack(outs).mean(dim=0)

    sub = pd.read_csv(SAMPLE_SUB)
    progress_csv = "/kaggle/working/submission_progress.csv"
    tag = "tta" if RUN_TTA_SUBMISSION else "fast"
    iter_csv = f"/kaggle/working/submission_iter_{TOTAL_ITER}_{tag}.csv"
    final_csv = "/kaggle/working/submission.csv"

    for i, fn in enumerate(tqdm(sub["filename"].tolist(), desc="Predicting test")):
        img = Image.open(os.path.join(TEST_LR, fn)).convert("RGB")
        arr = np.array(img).astype(np.float32) / 255.0
        x = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).cuda().float()
        y = predict_tensor(x)
        out_rgb = y.squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy()
        out_rgb = np.rint(out_rgb * 255.0).astype(np.uint8)
        out_to_encode = out_rgb[..., ::-1] if ENCODE_BGR else out_rgb
        sub.at[i, "rle"] = encode(out_to_encode)

        if (i + 1) % 200 == 0:
            sub.to_csv(progress_csv, index=False)

    sub.to_csv(iter_csv, index=False)
    sub.to_csv(final_csv, index=False)
    shutil.copy2(ckpt_path, "/kaggle/working/best_model_latest.pth")

    print("Saved session submission:", iter_csv)
    print("Saved Kaggle submission:", final_csv)
    print("Saved checkpoint copy:", "/kaggle/working/best_model_latest.pth")
    print("Missing rle:", sub["rle"].isna().sum())
    print("Empty rle:", (sub["rle"].astype(str).str.len() == 0).sum())
    print("rle starts with b':", sub["rle"].astype(str).str.startswith("b'").mean())
    print("Example rle:", sub.loc[0, "rle"])
    print(sub.head())
else:
    print(f"Inference skipped at {TOTAL_ITER}/{TOTAL_TARGET_ITER}.")
    print("Set RUN_INFERENCE_EACH_SESSION=True or FORCE_INFERENCE_NOW=True in Cell 0 to create submission.csv now.")